# Block A: Data Discovery & Validation

Objective: Understand the exact structure, availability, and quality of the CTU-CHB dataset before any processing.

This notebook:
- Scans all recordings
- Detects signal availability (FHR, UC, optional MHR)
- Extracts outcomes (pH, Apgar1/5) and derives Normal/Pathological labels when possible
- Computes basic signal quality metrics (missingness, outliers, valid %)
- Saves:
  - outputs/dataset_summary.csv
  - outputs/dataset_statistics.json


In [21]:
import os
import re
import json
import ast
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Try to import wfdb, install if not available
try:
    import wfdb
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "wfdb"])
    import wfdb

print("Libraries imported successfully")


Libraries imported successfully


In [22]:
# ----------------------------
# CONFIG / PATHS
# ----------------------------
data_dir = Path("../data/ctu-chb-intrapartum-cardiotocography-database-1.0.0").resolve()
out_dir = Path("../outputs").resolve()
out_dir.mkdir(parents=True, exist_ok=True)

MAX_RECORDS = None   # set to int for quick tests, or None for full scan

# Quality criterion (Block A)
MIN_VALID_FHR_PCT = 50.0  # pass if >= 50% of samples are valid FHR

# Physiologic ranges (outlier definition)
FHR_MIN, FHR_MAX = 80, 240
UC_MIN, UC_MAX = 0, 100   # broad range for discovery

print("data_dir:", data_dir)
print("out_dir:", out_dir)
print("Dataset exists:", data_dir.exists())


data_dir: /home/naem_haq/projects/CTG/data/ctu-chb-intrapartum-cardiotocography-database-1.0.0
out_dir: /home/naem_haq/projects/CTG/outputs
Dataset exists: True


In [23]:
hea_files = sorted(list(data_dir.glob("*.hea")))
print(f"Found {len(hea_files)} .hea files")

if len(hea_files) == 0:
    raise FileNotFoundError("No .hea files found. Check data_dir and extraction path.")

if MAX_RECORDS is not None:
    hea_files = hea_files[:MAX_RECORDS]
    print(f"Limiting scan to first {len(hea_files)} records")


Found 552 .hea files


In [24]:
def _norm_label(s: str) -> str:
    return re.sub(r"[^A-Z0-9]+", " ", str(s).upper()).strip()

FHR_KEYS = {"FHR", "FETAL HR", "FETALHEARTRATE", "FETAL HEART RATE"}
UC_KEYS  = {"UC", "TOCO", "UA", "UTERINE", "CONTRACTION", "CONTRACTIONS"}
MHR_KEYS = {"MHR", "MATERNAL HR", "MATERNALHEARTRATE", "MATERNAL HEART RATE"}

def detect_signals_from_header(hdr):
    """
    Detect FHR/UC/MHR from WFDB header sig_name (preferred), otherwise comments.
    Returns flags + debug labels list.
    """
    labels = []
    if getattr(hdr, "sig_name", None):
        labels = [_norm_label(x) for x in hdr.sig_name]
    else:
        labels = [_norm_label(c) for c in getattr(hdr, "comments", [])]

    found = {"fhr_available": False, "uc_available": False, "mhr_available": False}
    for lab in labels:
        if any(k in lab for k in FHR_KEYS): found["fhr_available"] = True
        if any(k in lab for k in UC_KEYS):  found["uc_available"]  = True
        if any(k in lab for k in MHR_KEYS): found["mhr_available"] = True

    return found, labels

print("Signal detection helpers ready")


Signal detection helpers ready


In [25]:
OUTCOME_PATTERNS = [
    (re.compile(r"\bREC\s*TYPE\b[:\s]+([NP]|NORMAL|PATHOLOGICAL|ABNORMAL)\b", re.I), "rec_type"),
    (re.compile(r"\bOUTCOME\b[:\s]+(NORMAL|PATHOLOGICAL|ABNORMAL)\b", re.I), "outcome"),
    (re.compile(r"\bPH\b[:\s]+([0-9]\.[0-9]+)\b", re.I), "pH"),
    (re.compile(r"\bAPGAR\s*1\b[:\s]+(\d+)\b", re.I), "apgar1"),
    (re.compile(r"\bAPGAR\s*5\b[:\s]+(\d+)\b", re.I), "apgar5"),
]

def parse_outcomes_from_comments(comments):
    outcomes = {}
    for c in comments:
        for rx, key in OUTCOME_PATTERNS:
            m = rx.search(str(c))
            if m and key not in outcomes:
                outcomes[key] = m.group(1)
    return outcomes

def outcome_label_from_outcomes(outcomes):
    """
    Priority:
      1) REC TYPE / OUTCOME strings
      2) Fallback: pH threshold (<7.15 pathological)
      3) else NaN
    """
    if "rec_type" in outcomes:
        v = str(outcomes["rec_type"]).strip().upper()
        if v in {"N", "NORMAL"}:
            return "Normal"
        if v in {"P", "PATHOLOGICAL", "ABNORMAL"}:
            return "Pathological"

    if "outcome" in outcomes:
        v = str(outcomes["outcome"]).strip().lower()
        if "normal" in v:
            return "Normal"
        if "path" in v or "abnormal" in v:
            return "Pathological"

    if "pH" in outcomes:
        ph = float(outcomes["pH"])
        return "Pathological" if ph < 7.15 else "Normal"

    return np.nan

print("Outcome parsing helpers ready")


Outcome parsing helpers ready


In [26]:
def compute_quality_metrics(fhr=None, uc=None):
    """
    Missingness/outliers:
      - FHR missing: NaN OR 0 (common CTG dropout encoding)
      - FHR outlier: outside [FHR_MIN, FHR_MAX] when not missing
      - UC missing: NaN (UC is often always present)
      - UC outlier: outside [UC_MIN, UC_MAX] when not missing
    """
    metrics = {
        "fhr_missing_pct": np.nan,
        "fhr_outliers_pct": np.nan,
        "fhr_valid_pct": np.nan,
        "uc_missing_pct": np.nan,
        "uc_outliers_pct": np.nan,
    }

    if fhr is not None:
        fhr = np.asarray(fhr, dtype=float)
        fhr_missing = np.isnan(fhr) | (fhr == 0)
        fhr_outlier = (~fhr_missing) & ((fhr < FHR_MIN) | (fhr > FHR_MAX))
        fhr_valid = (~fhr_missing) & (~fhr_outlier)

        metrics["fhr_missing_pct"] = 100.0 * fhr_missing.mean()
        metrics["fhr_outliers_pct"] = 100.0 * fhr_outlier.mean()
        metrics["fhr_valid_pct"] = 100.0 * fhr_valid.mean()

    if uc is not None:
        uc = np.asarray(uc, dtype=float)
        uc_missing = np.isnan(uc)
        uc_outlier = (~uc_missing) & ((uc < UC_MIN) | (uc > UC_MAX))

        metrics["uc_missing_pct"] = 100.0 * uc_missing.mean()
        metrics["uc_outliers_pct"] = 100.0 * uc_outlier.mean()

    return metrics

print("Quality metric helpers ready")


Quality metric helpers ready


In [27]:
test_rec = hea_files[0].with_suffix("")
hdr = wfdb.rdheader(str(test_rec))

found, labels = detect_signals_from_header(hdr)
outcomes = parse_outcomes_from_comments(getattr(hdr, "comments", []))
label = outcome_label_from_outcomes(outcomes)

print("Test record:", hdr.record_name)
print("n_sig:", hdr.n_sig, "fs:", hdr.fs, "sig_len:", hdr.sig_len)
print("Detected:", found)
print("sig_name:", getattr(hdr, "sig_name", None))
print("outcomes parsed:", outcomes)
print("label:", label)


Test record: 1001
n_sig: 2 fs: 4 sig_len: 19200
Detected: {'fhr_available': True, 'uc_available': True, 'mhr_available': False}
sig_name: ['FHR', 'UC']
outcomes parsed: {'pH': '7.14', 'apgar1': '6', 'apgar5': '8'}
label: Pathological


In [28]:
all_records = []
signal_inventory = defaultdict(int)
read_failures = []

quality_pass = 0
quality_fail = 0

for hea_path in hea_files:
    rec_path = hea_path.with_suffix("")  # without extension
    rec_id = rec_path.name

    row = {
        "record_id": rec_id,
        "num_signals": np.nan,
        "sampling_rate": np.nan,
        "num_samples": np.nan,
        "duration_seconds": np.nan,
        "duration_minutes": np.nan,
        "fhr_available": False,
        "uc_available": False,
        "mhr_available": False,
        "outcome_label": np.nan,
        "outcomes_raw": {},
        "sig_names_dbg": [],
        "fhr_missing_pct": np.nan,
        "fhr_outliers_pct": np.nan,
        "fhr_valid_pct": np.nan,
        "uc_missing_pct": np.nan,
        "uc_outliers_pct": np.nan,
    }

    try:
        hdr = wfdb.rdheader(str(rec_path))
        row["num_signals"] = int(hdr.n_sig)
        row["sampling_rate"] = float(hdr.fs)
        row["num_samples"] = int(hdr.sig_len)
        row["duration_seconds"] = float(hdr.sig_len / hdr.fs) if hdr.fs else np.nan
        row["duration_minutes"] = float(row["duration_seconds"] / 60.0) if row["duration_seconds"] else np.nan

        found, labels = detect_signals_from_header(hdr)
        row.update(found)
        row["sig_names_dbg"] = [str(x) for x in getattr(hdr, "sig_name", labels)]

        outcomes = parse_outcomes_from_comments(getattr(hdr, "comments", []))
        row["outcomes_raw"] = outcomes
        row["outcome_label"] = outcome_label_from_outcomes(outcomes)

        signal_inventory["Total"] += 1
        if row["fhr_available"]: signal_inventory["FHR"] += 1
        if row["uc_available"]:  signal_inventory["UC"] += 1
        if row["mhr_available"]: signal_inventory["MHR"] += 1

        # Read samples for quality metrics
        p_signal, fields = wfdb.rdsamp(str(rec_path))
        sig_names = fields.get("sig_name", [])
        sig_names_norm = [_norm_label(n) for n in sig_names]

        fhr_idx, uc_idx = None, None
        for i, nlab in enumerate(sig_names_norm):
            if fhr_idx is None and any(k in nlab for k in FHR_KEYS):
                fhr_idx = i
            if uc_idx is None and any(k in nlab for k in UC_KEYS):
                uc_idx = i

        fhr = p_signal[:, fhr_idx] if fhr_idx is not None else None
        uc  = p_signal[:, uc_idx] if uc_idx is not None else None

        qm = compute_quality_metrics(fhr=fhr, uc=uc)
        row.update(qm)

        # Quality pass/fail
        if pd.notna(row["fhr_valid_pct"]):
            if float(row["fhr_valid_pct"]) >= MIN_VALID_FHR_PCT:
                quality_pass += 1
            else:
                quality_fail += 1

    except Exception as e:
        read_failures.append({"record_id": rec_id, "error": str(e)})

    all_records.append(row)

print(f"Scanned {len(all_records)} records")
print(f"Read failures: {len(read_failures)}")
if read_failures[:3]:
    print("Example failures:", read_failures[:3])


Scanned 552 records
Read failures: 0


In [29]:
df = pd.DataFrame(all_records)

print("=" * 80)
print("SIGNAL INVENTORY")
print("=" * 80)

total = signal_inventory.get("Total", len(df))
for k in ["FHR", "UC", "MHR"]:
    if k in signal_inventory:
        pct = 100 * signal_inventory[k] / total if total else 0
        print(f"  {k:10s}: {signal_inventory[k]:4d} / {total} ({pct:5.1f}%)")

print("\n" + "=" * 80)
print("RECORDING DURATION STATISTICS")
print("=" * 80)
print(f"  Mean:   {df['duration_minutes'].mean():.2f} min")
print(f"  Median: {df['duration_minutes'].median():.2f} min")
print(f"  Std:    {df['duration_minutes'].std():.2f} min")
print(f"  Min:    {df['duration_minutes'].min():.2f} min")
print(f"  Max:    {df['duration_minutes'].max():.2f} min")
print(f"  Total:  {df['duration_minutes'].sum():.0f} min ({df['duration_minutes'].sum()/60:.1f} hours)")

print("\n" + "=" * 80)
print("OUTCOME LABEL DISTRIBUTION")
print("=" * 80)
counts = df["outcome_label"].value_counts(dropna=False)
for outcome, count in counts.items():
    name = "Unknown/Missing" if pd.isna(outcome) else str(outcome)
    pct = 100 * count / len(df) if len(df) else 0
    print(f"  {name:20s}: {count:4d} ({pct:5.1f}%)")


SIGNAL INVENTORY
  FHR       :  552 / 552 (100.0%)
  UC        :  552 / 552 (100.0%)

RECORDING DURATION STATISTICS
  Mean:   74.16 min
  Median: 71.71 min
  Std:    7.56 min
  Min:    60.00 min
  Max:    90.08 min
  Total:  40938 min (682.3 hours)

OUTCOME LABEL DISTRIBUTION
  Normal              :  447 ( 81.0%)
  Pathological        :  103 ( 18.7%)
  Unknown/Missing     :    2 (  0.4%)


In [30]:
# ============================
# Clean + Save Outputs
# - outcomes_raw -> ph/apgar1/apgar5
# - blank outcome_label -> NaN
# - add label_source
# - add quality_pass flag
# - write dataset_summary.csv + dataset_statistics.json
# ============================

# 1) Normalize outcome_label (blank -> NaN)
df["outcome_label"] = df["outcome_label"].astype("string")
df["outcome_label"] = df["outcome_label"].replace({"": pd.NA, "None": pd.NA, "nan": pd.NA})
df.loc[df["outcome_label"].str.strip().isna(), "outcome_label"] = pd.NA

# 2) Ensure outcomes_raw is a dict (not a stringified dict)
def _to_dict(x):
    if isinstance(x, dict):
        return x
    if pd.isna(x):
        return {}
    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return {}
        try:
            return json.loads(s)
        except Exception:
            try:
                return ast.literal_eval(s)
            except Exception:
                return {}
    return {}

df["outcomes_raw"] = df["outcomes_raw"].apply(_to_dict)

# 3) Extract ph/apgar1/apgar5 columns
def _safe_float(v):
    try:
        if v is None:
            return np.nan
        return float(str(v).strip())
    except Exception:
        return np.nan

def _safe_int(v):
    try:
        if v is None:
            return np.nan
        return int(float(str(v).strip()))
    except Exception:
        return np.nan

df["ph"] = df["outcomes_raw"].apply(lambda d: _safe_float(d.get("pH", d.get("ph"))))
df["apgar1"] = df["outcomes_raw"].apply(lambda d: _safe_int(d.get("apgar1", d.get("Apgar1", d.get("APGAR1")))))
df["apgar5"] = df["outcomes_raw"].apply(lambda d: _safe_int(d.get("apgar5", d.get("Apgar5", d.get("APGAR5")))))

# 4) Add label_source
def _label_source(row):
    d = row["outcomes_raw"] if isinstance(row["outcomes_raw"], dict) else {}
    if "rec_type" in d or "REC TYPE" in d:
        return "rec_type"
    if "outcome" in d or "Outcome" in d or "OUTCOME" in d:
        return "outcome"
    if pd.notna(row["outcome_label"]) and pd.notna(row["ph"]):
        return "ph_fallback"
    return "missing"

df["label_source"] = df.apply(_label_source, axis=1)

# 5) Add quality_pass boolean
df["quality_pass"] = df["fhr_valid_pct"].apply(lambda x: bool(pd.notna(x) and float(x) >= MIN_VALID_FHR_PCT))

# 6) Recompute inventory + summary stats from cleaned df
signal_inventory_clean = {
    "Total": int(len(df)),
    "FHR": int(df["fhr_available"].sum()),
    "UC": int(df["uc_available"].sum()),
}
if "mhr_available" in df.columns:
    signal_inventory_clean["MHR"] = int(df["mhr_available"].sum())

quality_pass_n = int(df["quality_pass"].sum())
quality_fail_n = int((~df["quality_pass"]).sum())

# 7) Save cleaned dataset_summary.csv
summary_csv_path = out_dir / "dataset_summary.csv"
df.to_csv(summary_csv_path, index=False)

# 8) Save cleaned dataset_statistics.json (same format you posted)
stats = {
    "total_recordings": int(len(df)),
    "signal_inventory": signal_inventory_clean,
    "duration_minutes": {
        "mean": float(df["duration_minutes"].mean()),
        "median": float(df["duration_minutes"].median()),
        "std": float(df["duration_minutes"].std()),
        "min": float(df["duration_minutes"].min()),
        "max": float(df["duration_minutes"].max()),
        "total_hours": float(df["duration_minutes"].sum() / 60.0),
    },
    "class_distribution": {
        "Normal": int((df["outcome_label"] == "Normal").sum()),
        "Pathological": int((df["outcome_label"] == "Pathological").sum()),
        "Unknown": int(df["outcome_label"].isna().sum()),
    },
    "quality": {
        "criteria": f"valid_fhr_pct >= {MIN_VALID_FHR_PCT:.1f}",
        "pass": int(quality_pass_n),
        "fail": int(quality_fail_n),
        "mean_fhr_missing_pct": float(df["fhr_missing_pct"].mean()),
        "mean_fhr_outliers_pct": float(df["fhr_outliers_pct"].mean()),
        "mean_uc_missing_pct": float(df["uc_missing_pct"].mean()),
        "mean_uc_outliers_pct": float(df["uc_outliers_pct"].mean()),
    },
    "read_failures": read_failures,
}

stats_json_path = out_dir / "dataset_statistics.json"
with open(stats_json_path, "w") as f:
    json.dump(stats, f, indent=2)

print("Saved cleaned outputs:")
print(" -", summary_csv_path)
print(" -", stats_json_path)

df[["record_id", "outcome_label", "label_source", "ph", "apgar1", "apgar5",
    "fhr_missing_pct", "fhr_outliers_pct", "fhr_valid_pct", "quality_pass"]].head(10)


Saved cleaned outputs:
 - /home/naem_haq/projects/CTG/outputs/dataset_summary.csv
 - /home/naem_haq/projects/CTG/outputs/dataset_statistics.json


,record_id,outcome_label,label_source,ph,apgar1,apgar5,fhr_missing_pct,fhr_outliers_pct,fhr_valid_pct,quality_pass
0,1001,Pathological,ph_fallback,7.14,6,8,22.161458,1.140625,76.697917,True
1,1002,<NA>,missing,NaN,8,8,16.984375,2.510417,80.505208,True
2,1003,Normal,ph_fallback,7.20,7,9,20.477778,3.027778,76.494444,True
3,1004,Normal,ph_fallback,7.30,8,9,1.339286,3.517857,95.142857,True
4,1005,Normal,ph_fallback,7.30,9,10,22.016667,2.561111,75.422222,True
5,1006,Normal,ph_fallback,7.23,8,9,24.339286,1.666667,73.994048,True
6,1007,Normal,ph_fallback,7.16,9,10,21.743590,3.673077,74.583333,True
7,1008,Normal,ph_fallback,7.36,8,9,8.934524,0.071429,90.994048,True
8,1009,Normal,ph_fallback,7.18,8,9,34.151961,1.715686,64.132353,True
9,1010,Normal,ph_fallback,7.35,8,9,11.267857,1.970238,86.761905,True


In [35]:
unknown_ids = df.loc[df["outcome_label"].isna(), "record_id"].astype(str).tolist()
fail_ids = df.loc[~df["quality_pass"], "record_id"].astype(str).tolist()

# Update stats dict before dumping
stats["label_source_counts"] = {
    k: int(v) for k, v in df["label_source"].value_counts(dropna=False).to_dict().items()
}
stats["unknown_record_ids"] = unknown_ids
stats["failed_quality_record_ids"] = fail_ids

# --- Add labeling rule metadata ---
stats["labeling"] = {
    "primary": "REC TYPE / OUTCOME if present",
    "fallback": "pH threshold",
    "ph_threshold_pathological": 7.15
}

# --- Add quality distribution stats (median + IQR) ---
def _qstats(series: pd.Series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return {"median": None, "p25": None, "p75": None}
    return {
        "median": float(s.median()),
        "p25": float(s.quantile(0.25)),
        "p75": float(s.quantile(0.75)),
    }

stats["quality_distributions"] = {
    "fhr_missing_pct": _qstats(df["fhr_missing_pct"]),
    "fhr_valid_pct": _qstats(df["fhr_valid_pct"]),
    "fhr_outliers_pct": _qstats(df["fhr_outliers_pct"]),
    "uc_missing_pct": _qstats(df["uc_missing_pct"]),
    "uc_outliers_pct": _qstats(df["uc_outliers_pct"]),
}

with open(stats_json_path, "w") as f:
    json.dump(stats, f, indent=2)

print("Patched JSON with provenance lists + labeling + distributions:")
print(" - unknown_record_ids:", unknown_ids)
print(" - failed_quality_record_ids (n):", len(fail_ids))
print(" - quality_distributions keys:", list(stats["quality_distributions"].keys()))


Patched JSON with provenance lists + labeling + distributions:
 - unknown_record_ids: ['1002', '1017']
 - failed_quality_record_ids (n): 10
 - quality_distributions keys: ['fhr_missing_pct', 'fhr_valid_pct', 'fhr_outliers_pct', 'uc_missing_pct', 'uc_outliers_pct']


## Notes

- FHR missingness treats `FHR == 0` as dropout (plus NaNs).
- FHR outliers are values outside [80, 240] bpm after excluding missing.
- Outcome label is derived from:
  1) `REC TYPE` / `OUTCOME` (if present)
  2) otherwise pH fallback (<7.15 -> Pathological)
  3) otherwise Unknown (NaN)

Outputs are saved into `../outputs/`.


In [36]:
# Provenance / sanity report
print("\nLABEL SOURCE COUNTS")
print(df["label_source"].value_counts(dropna=False))

print("\nUNKNOWN LABEL RECORDS (first 20)")
print(df.loc[df["outcome_label"].isna(), ["record_id", "ph", "apgar1", "apgar5", "outcomes_raw"]].head(20))

print("\nFAILED QUALITY RECORDS (first 20)")
print(df.loc[~df["quality_pass"], ["record_id", "fhr_valid_pct", "fhr_missing_pct", "fhr_outliers_pct"]].head(20))



LABEL SOURCE COUNTS
label_source
ph_fallback    550
missing          2
Name: count, dtype: int64

UNKNOWN LABEL RECORDS (first 20)
   record_id  ph  apgar1  apgar5                    outcomes_raw
1       1002 NaN       8       8  {'apgar1': '8', 'apgar5': '8'}
16      1017 NaN       6       7  {'apgar1': '6', 'apgar5': '7'}

FAILED QUALITY RECORDS (first 20)
    record_id  fhr_valid_pct  fhr_missing_pct  fhr_outliers_pct
57       1058      47.694444        51.648148          0.657407
163      1164      48.711111        49.083333          2.205556
172      1173      49.911458        49.723958          0.364583
197      1198      43.403846        46.057692         10.538462
360      1361      42.678571        47.119048         10.202381
430      1431      48.101828        51.815642          0.082529
511      2006      47.678949        52.278808          0.042244
518      2013      45.923795        53.256622          0.819583
530      2025      46.497648        53.502352          0.00000